# BDD-OIA

This notebook shows an implementation of [BDD-OIA](https://twizwei.github.io/bddoia_project/) task. The BDD-OIA dataset comprises frames extracted from driving scene videos, which are used for autonomous driving predictions. Each frame is annotated with 4 binary labels, indicating the possible actions, namely $\textsf{move\_forward}$, $\textsf{stop}$, $\textsf{turn\_left}$, $\textsf{turn\_right}$. Each frame is also annotated with 21 intermediate binary concepts such as $\textsf{red\_light}$, $\textsf{road\_clear}$, etc., underlying the reasons for the possible actions.

The objective is to predict possible actions for each frame. During training, we only make use of the label supervision, along with a knowledge base, which comprises information about the relations between concepts and actions, e.g., $\textsf{red\_light} \lor \textsf{traffic\_sign} \lor \textsf{obstacle} \implies \textsf{stop}$. The training set consists of 16,000 frames, while the test set contains 4,500 annotated data points.

Before usage, the dataset was pre-processed by [Marconato et al. (2023)](https://proceedings.neurips.cc/paper_files/paper/2023/file/e560202b6e779a82478edb46c6f8f4dd-Paper-Conference.pdf) using a pretrained Faster-RCNN model on BDD-100k, in conjunction with the first module in CBM-AUC [(Sawada & Nakamura, 2022)](https://arxiv.org/abs/2202.01459), resulting in embeddings of dimension 2048.

In [1]:
# Import necessary libraries and modules
import os.path as osp  

import numpy as np
import torch
import torch.nn as nn
from torch import optim

from ablkit.data.evaluation import SymbolAccuracy
from ablkit.reasoning import Reasoner
from ablkit.utils import ABLLogger, print_log

from models.nn import ConceptNet
from models.bdd_nn import BDDNN
from models.bdd_model import BDDABLModel
from ablkit.reasoning import KBBase
from dataset.data_util import get_dataset
from bridge import BDDBridge
from metric import BDDReasoningMetric

## Working with Data

First, we get the training and testing datasets:

In [2]:
train_data = get_dataset(fname="train.npz", get_pseudo_label=True)
val_data = get_dataset(fname="val.npz", get_pseudo_label=True)
test_data = get_dataset(fname="test.npz", get_pseudo_label=True)

In [3]:
print(f"Both train_data and test_data consist of 3 components: X, gt_pseudo_label, Y")
print()
train_X, train_gt_pseudo_label, train_Y = train_data
print(
    f"Length of X, gt_pseudo_label, Y in train_data: "
    + f"{len(train_X)}, {len(train_gt_pseudo_label)}, {len(train_Y)}"
)
test_X, test_gt_pseudo_label, test_Y = test_data
print(
    f"Length of X, gt_pseudo_label, Y in test_data: "
    + f"{len(test_X)}, {len(test_gt_pseudo_label)}, {len(test_Y)}"
)
print()

X_0, gt_pseudo_label_0, Y_0 = train_X[0], train_gt_pseudo_label[0], train_Y[0]
print(
    f"X is a {type(train_X).__name__}, "
    + f"with each element being a {type(X_0).__name__} of {type(X_0[0]).__name__}."
)
print(
    f"gt_pseudo_label is a {type(train_gt_pseudo_label).__name__}, "
    + f"with each element being a {type(gt_pseudo_label_0).__name__} "
    + f"of {type(gt_pseudo_label_0[0]).__name__}."
)
print(f"Y is a {type(train_Y).__name__}, " + f"with each element being an {type(Y_0).__name__}.")

Both train_data and test_data consist of 3 components: X, gt_pseudo_label, Y

Length of X, gt_pseudo_label, Y in train_data: 16082, 16082, 16082
Length of X, gt_pseudo_label, Y in test_data: 4572, 4572, 4572

X is a list, with each element being a list of ndarray.
gt_pseudo_label is a list, with each element being a list of int.
Y is a list, with each element being an tuple.


## Building the Learning Part

To build the learning part, we need to first build a machine learning base model. We use ConceptNet, and encapsulate it within a `BDDNN` object to create the base model. `BDDNN` is a class that encapsulates a PyTorch model, transforming it into a base model with an sklearn-style interface.

In [4]:
cls = ConceptNet()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(cls.parameters(), lr=0.002)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.002,
    pct_start=0.15,
    epochs=2,
    steps_per_epoch=int(1 / 0.01) + 1,
)

base_model = BDDNN(
    cls,
    loss_fn,
    optimizer,
    scheduler=scheduler,
    device=device,
    batch_size=32,
    num_epochs=1,
)

However, the base model built above deals with instance-level data, and can not directly deal with example-level data. Therefore, we wrap the base model into `BDDABLModel`, a subclass of `ABLModel`, which enables the learning part to train, test, and predict on example-level data.

In [5]:
model = BDDABLModel(base_model)

## Building the Reasoning Part

In the reasoning part, we first build a knowledge base which contains information about the relations between concepts and actions, e.g., $\text{red\_light} \lor \text{traffic\_sign} \lor \text{obstacle} \implies \text{stop}$. The knowledge base is built in the `BDDKB` class within file `reasoning/bddkb.py`, and is derived from the `KBBase` class. In the derived subclass, we initialize the `pseudo_label_list` parameter specifying list of possible pseudo-labels, and override the `logic_forward` function defining how to perform (deductive) reasoning.

In [6]:
class BDDKB(KBBase):
    def __init__(self, pseudo_label_list=None):
        if pseudo_label_list is None:
            pseudo_label_list = [0, 1]
        super().__init__(pseudo_label_list)

    def logic_forward(self, attrs):
        """
        Abduction space
        (0, 1, 0, 0) 610812
        (0, 1, 0, 1) 75012
        (0, 1, 1, 0) 75012
        (0, 1, 1, 1) 9212
        (1, 0, 0, 0) 12996
        (1, 0, 0, 1) 1596
        (1, 0, 1, 0) 1596
        (1, 0, 1, 1) 196
        """
        assert len(attrs) == 21
        (
            green_light,
            follow,
            road_clear,
            red_light,
            traffic_sign,
            car,
            person,
            rider,
            other_obstacle,
            left_lane,
            left_green_light,
            left_follow,
            no_left_lane,
            left_obstacle,
            left_solid_line,
            right_lane,
            right_green_light,
            right_follow,
            no_right_lane,
            right_obstacle,
            right_solid_line,
        ) = attrs

        illegal_return = (0, 0, 0, 0)
        if red_light == green_light == 1:
            return illegal_return
        obstacle = car or person or rider or other_obstacle
        if road_clear == obstacle:
            return illegal_return
        move_forward = green_light or follow or road_clear
        stop = red_light or traffic_sign or obstacle
        if stop:
            move_forward = 0

        can_turn_left = left_lane or left_green_light or left_follow
        cannot_turn_left = no_left_lane or left_obstacle or left_solid_line
        turn_left = can_turn_left and int(not cannot_turn_left)

        can_turn_right = right_lane or right_green_light or right_follow
        cannot_turn_right = no_right_lane or right_obstacle or right_solid_line
        turn_right = can_turn_right and int(not cannot_turn_right)

        return move_forward, stop, turn_left, turn_right

kb = BDDKB()

Then, we create a reasoner by instantiating the class `Reasoner`. Due to the indeterminism of abductive reasoning, there could be multiple candidates compatible with the knowledge base. When this happens, reasoner can minimize inconsistencies between the knowledge base and pseudo-labels predicted by the learning part, and then return only one candidate that has the highest consistency. The inconsistency is calculated by the function function `multi_label_confidence_dist`.

In [7]:
def multi_label_confidence_dist(data_example, candidates, candidates_idxs, reasoning_results):
    pred_prob = data_example.pred_prob.T  # nc x 1
    pred_prob = np.concatenate([1 - pred_prob, pred_prob], axis=1)  # nc x 2
    cols = np.arange(len(candidates_idxs[0]))[None, :]
    corr_prob = pred_prob[cols, candidates_idxs]
    costs = -np.sum(np.log(corr_prob + 1e-6), axis=1)
    return costs

reasoner = Reasoner(
    kb,
    dist_func=multi_label_confidence_dist,
    max_revision=3,
    require_more_revision=3,
)

## Building Evaluation Metrics

Next, we set up evaluation metrics. These metrics will be used to evaluate the model performance during training and testing. Specifically, we use `SymbolAccuracy` and `BDDReasoningMetric`, which are used to evaluate the accuracy of the machine learning model’s predictions and the accuracy of the final reasoning results, respectively.

In [8]:
metric_list = [SymbolAccuracy(prefix="bdd_oia"), BDDReasoningMetric(kb=kb, prefix="bdd_oia")]

## Bridging Learning and Reasoning

Now, the last step is to bridge the learning and reasoning part. We proceed with this step by creating an instance of `BDDBridge`, which is derived from `SimpleBridge`.

In [9]:
bridge = BDDBridge(model, reasoner, metric_list)

Perform training and testing by invoking the `train` and `test` methods of `SimpleBridge`.

In [10]:
# Build logger
print_log("Abductive Learning on the MNIST Addition example.", logger="current")
log_dir = ABLLogger.get_current_instance().log_dir
weights_dir = osp.join(log_dir, "weights")

bridge.train(train_data, loops=2, segment_size=0.01, save_interval=1, save_dir=weights_dir)
bridge.test(test_data)

03/03 17:16:56 - abl - INFO - Abductive Learning on the MNIST Addition example.
03/03 17:16:56 - abl - INFO - loop(train) [1/2] segment(train) [1/101] 
03/03 17:16:58 - abl - INFO - model loss: 0.60486
03/03 17:16:58 - abl - INFO - loop(train) [1/2] segment(train) [2/101] 
03/03 17:17:01 - abl - INFO - model loss: 0.59258
03/03 17:17:01 - abl - INFO - loop(train) [1/2] segment(train) [3/101] 
03/03 17:17:03 - abl - INFO - model loss: 0.58585
03/03 17:17:03 - abl - INFO - loop(train) [1/2] segment(train) [4/101] 
03/03 17:17:05 - abl - INFO - model loss: 0.56797
03/03 17:17:05 - abl - INFO - loop(train) [1/2] segment(train) [5/101] 
03/03 17:17:07 - abl - INFO - model loss: 0.55167
03/03 17:17:07 - abl - INFO - loop(train) [1/2] segment(train) [6/101] 
03/03 17:17:08 - abl - INFO - model loss: 0.51662
03/03 17:17:08 - abl - INFO - loop(train) [1/2] segment(train) [7/101] 
03/03 17:17:09 - abl - INFO - model loss: 0.46525
03/03 17:17:09 - abl - INFO - loop(train) [1/2] segment(train) [8/